# Interactive version of biv-me
(edited version from original main_interactive implementation)

This jupyter notebook contains an identical implementation of biv-me to the main.py. The only difference is that it can be run step-by-step, allowing you to make manual corrections if and when needed to both the view selection and segmentation steps, which can be prone to occasional errors.

The only thing you will need on top of the regular biv-me environment is to download a Python IDE which can run Jupyter notebooks (e.g. Visual Studio Code), and install the ipython kernel as below.

```
conda install -n bivme311 ipykernel --update-deps --force-reinstall
```

When you run the notebook, you will be prompted to select the kernel to use. Make sure to select your bivme311 environment.

### Troubleshooting model performance
Though we are confident in the robustness of our deep learning models, they may not work perfectly for your data. If you find that the segmentation or view selection models perform poorly for your data, reach out to us at [joshua.dillon@auckland.ac.nz](joshua.dillon@auckland.ac.nz) or [charlene.1.mauger@kcl.ac.uk](charlene.1.mauger@kcl.ac.uk) and let us know what kind of data you are using. We are actively developing these models and always looking for ways to enhance their generalisability across vendors, protocols, centres, and patient demographics.


In [28]:
# Import libraries
import os,sys
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

import torch
import tomli
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))
import shutil
import time
import datetime
from loguru import logger

import warnings
warnings.filterwarnings('ignore')

In [29]:
# This allows for modules to be reloaded automatically if you make any changes to those files.
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [30]:
# Import modules
from bivme.preprocessing.dicom.extract_cines import extract_cines
from bivme.preprocessing.dicom.select_views import select_views
from bivme.preprocessing.dicom.segment_views import segment_views
from bivme.preprocessing.dicom.correct_phase_mismatch import correct_phase_mismatch
from bivme.preprocessing.dicom.generate_contours import generate_contours
from bivme.preprocessing.dicom.export_guidepoints import export_guidepoints
from bivme.plotting.plot_guidepoints import generate_html 
from main import run_fitting

In [31]:
# Set up logging
log_level = "DEBUG"
log_format = "<green>{time:YYYY-MM-DD HH:mm:ss.SSS zz}</green> | <level>{level: <8}</level> | <yellow>Line {line: >4} ({file}):</yellow> <b>{message}</b>"

In [32]:
# Check if GPU is available (torch)
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")
    logger.warning('No GPU available. Using CPU instead. This may be very slow!')
    
logger.info(f"Using device: {device}")

2026-06-27 10:14:03.242 | WARNING  | __main__:<module>:8 - No GPU available. Using CPU instead. This may be very slow!
2026-06-27 10:14:03.243 | INFO     | __main__:<module>:10 - Using device: cpu


# Config file
As with the main.py, you should customise the template config file in `src/bivme/configs/config.toml` for your specific batch and use case. This config file will be read in and the inputs used for the remainder of this pipeline.

In [ ]:
config_file = './bivme/configs/config_sunnybrook_defv4.toml' # Path to the config file

In [34]:
# Load config
with open(config_file, mode="rb") as fp:
    logger.info(f'Loading config file: {config_file}')
    config = tomli.load(fp)

# TOML Schema Validation
# This needs to be done to make sure the config file is valid.
# Going forward, we will trust you that all of the inputs in the config file are correct, unlike the main.py which does not trust you.

match config:
    case {
        "modules": {"preprocessing": bool(), "fitting": bool()},

        "logging": {"show_detailed_logging": bool(), "generate_log_file": bool()},

        "plotting": {"generate_plots_preprocessing": bool(), "generate_plots_fitting": bool(), "include_images": bool(), "export_images": bool()},

        "input_pp": {"source": str(),
                    "batch_ID": str(),
                    "analyst_id": str(),
                    "processing": str(),
                    "states": str()
                    },
        "view-selection": {"option": str(), "correct_mode": str()},
        "contouring": {"smooth_landmarks": bool()},
        "output_pp": {"overwrite": bool(), "output_directory": str()},

        "input_fitting": {"gp_directory": str(),
                    "gp_suffix": str(),
                    "si_suffix": str(),
                    },
        "breathhold_correction": {"shifting": str(), "ed_frame": int()},
        "gp_processing": {"sampling": int(), "num_of_phantom_points_av": int(), "num_of_phantom_points_mv": int(), "num_of_phantom_points_tv": int(), "num_of_phantom_points_pv": int()},
        "multiprocessing": {"workers": int()},
        "fitting_weights": {"guide_points": float(), "convex_problem": float(), "transmural": float()},
        "output_fitting": {"output_directory": str(), "output_meshes": list(), "closed_mesh": bool(),   "export_control_mesh": bool(), "mesh_format": str(),  "overwrite": bool()},
    }:
        pass
    case _:
            raise ValueError(f"Invalid configuration: {config}")

2026-06-27 10:14:06.377 | INFO     | __main__:<module>:3 - Loading config file: config_sunnybrook_testv4.toml


# Start of preprocessing
This code reads in DICOM files and generates GPFiles for personalised biventricular mesh fitting.

## Step 0: Set up directories
All directories will be created for you except for the input_path. This should point to your DICOMs, separated into folders by case like so:

    input_path
    └─── case1
        │─── *
    └─── case2
        │─── *
    └─── ...

Don't worry about preprocessing your dicoms, separating by scan type, or excluding non-cines. The pipeline should find which ones are cines and which ones aren't by checking key terms within the series descriptions. Check src/bivme/preprocessing/dicom/extract_cines.py for the list of key terms.

In [35]:
batch_ID = config['input_pp']['batch_ID'] # This will serve as your output folder name. Example: 'test'
analyst_id = config['input_pp']['analyst_id'] # Example: 'analyst1'
input_path = config['input_pp']['source'] # Path to the input DICOM folder
processed_path = config['input_pp']['processing'] # Path to the processed folder, where view predictions and segmentations will be stored. This will be created upon run time.
states_path = config['input_pp']['states'] # Path to the states folder, where the logs and view predictions will be stored for reference. This will be created upon run time.
output_path = config['output_pp']['output_directory'] # Path to the output folder, where GP files will be stored. This will be created upon run time.
plotting_path = processed_path # Path to the plotting folder, where the HTML file of plotted guidepoints will be stored. This will be created upon run time.

# Target path: src/bivme/preprocessing/dicom/models
# You can hardcode the model path here or set it as an environment variable.
cwd = os.getcwd()
print('Current working directory:', cwd)
MODEL_DIR = os.path.join(cwd,'bivme/preprocessing/dicom/models')
print('Model directory:', MODEL_DIR)

Current working directory: /biv-me/src
Model directory: /biv-me/src/bivme/preprocessing/dicom/models


## Step 0.1: Choose case

In [36]:
case = 'sb3901' # Enter the case ID. This should be a subfolder within the input_path. 

In [37]:
# Set up batch specific folders
src = os.path.join(input_path)
dst = os.path.join(processed_path, batch_ID)
states = os.path.join(states_path, batch_ID)
output = os.path.join(output_path, batch_ID)
plotting = os.path.join(plotting_path, batch_ID)


case_src = os.path.join(src, case)
if not os.path.isdir(case_src):
    logger.error(f'Case {case} not found in source folder. Please check the case ID.')
    sys.exit()

os.makedirs(dst, exist_ok=True)
os.makedirs(states, exist_ok=True)
os.makedirs(output, exist_ok=True)
os.makedirs(plotting, exist_ok=True)

case_dst = os.path.join(dst, case)
case_output = os.path.join(output, case)

if os.path.exists(case_output):
    overwrite = config['output_pp']['overwrite']
    if overwrite:
        shutil.rmtree(case_dst)
        shutil.rmtree(case_output)
    else:
        print(f'This case has already been processed and you have set overwrite to false in the config file. Change the case ID or delete the existing folder {case_dst} before proceeding.')
        sys.exit()

elif os.path.exists(case_dst):
    overwrite = config['output_pp']['overwrite']
    if overwrite:
        shutil.rmtree(case_dst)
    else:
        print(f'This case has already been processed and you have set overwrite to false in the config file. Change the case ID or delete the existing folder {case_dst} before proceeding.')
        sys.exit()

In [38]:
# Create log file to record some details
states = os.path.join(states, case, analyst_id)
os.makedirs(states, exist_ok=True)

logger_id = logger.add(f'{states}/log_file_{datetime.datetime.now().strftime("%Y-%m-%d-%H-%M-%S")}.log', level=log_level, format=log_format,
    colorize=False, backtrace=True,
    diagnose=True)

logger.info(f'Processing case: {case}')

start_time = time.time()

2026-06-27 10:14:15.283 | INFO     | __main__:<module>:9 - Processing case: sb3901


## Step 0.2: Pre-preprocess dicoms
By default, the pipeline takes in only cardiac cine images in .dcm format. The pre-preprocessing reads in the raw dicoms and uses the series descriptions to infer which are cines and which aren't. The 'cine only' dicoms are saved to:

    processed_path   
    └───batch_ID
        └───processed-dicoms

In [39]:
logger.info(f'Finding cines...')
extract_cines(case_src, case_dst, logger)

case_src = os.path.join(case_dst, 'processed-dicoms') # Update source directory

logger.success(f'Pre-preprocessing complete. Cines extracted to {case_src}.')

2026-06-27 10:14:17.088 | INFO     | __main__:<module>:1 - Finding cines...
2026-06-27 10:14:17.875 | INFO     | bivme.preprocessing.dicom.extract_cines:extract_cines:45 - Found 260 images in the source directory
2026-06-27 10:14:17.877 | INFO     | bivme.preprocessing.dicom.extract_cines:extract_cines:46 - Extracted 260 which matched the inclusion criteria.
2026-06-27 10:14:18.231 | SUCCESS  | __main__:<module>:6 - Pre-preprocessing complete. Cines extracted to ../sunnybrook/processing/testsb10/sb3901/processed-dicoms.


## Step 1: View selection
View selection can be carried out in three main ways. 

The first option is the **'default'** option. This involves the use of two models - one using only dicom metadata and one using only image data. The metadata predictions are only broad categories (SAX, LAX, or Outflow) and are used to correct image-based predictions. This should be the most robust option, but takes longer to complete. 

The second option is the **'image-only'** option. This only uses image data. Use this if the metadata step is adding unnecessary time, or is not performing as expected.

The third option is the **'load'** option. This loads view predictions from the states folder. This is useful if view predictions have already been made, avoiding the need to rerun prediction from scratch. It also allows for you to make and save manual corrections to the selected views. 

On top of these options, you can also manually review and correct the selected views using a simple GUI. There are three options - **'manual'**, which always launches the GUI after view selection, **'adaptive'**, which only launches the GUI if there are low confidence predictions or missing views, or **'automatic'**, which does not launch the GUI and keeps the automatic predictions as they are.

#### Important note about required views
To proceed with fitting, you must have at least some SAX images and a 4ch image. This is because the fitting requires landmarks from the 4ch image to initialise the model (mitral valve, tricuspid valve, and apex point). The SAX images are required, along with at least one long axis image e.g. 4ch, to correct for breath-hold misalignment. Using the GUI on **'adaptive'** mode will make sure that you never miss a required view. 

In [40]:
option = 'default' # Either 'default', 'image-only', or 'load'. 
# 'default' will combine dicom metadata and image data to select the correct view. 'image-only' will only use the image data. 
# 'load' will load view predictions from the states folder, if view predictions have already been made.

correct_mode = 'adaptive' # Either 'manual', 'adaptive', or 'automatic'. 
# 'manual' will launch a GUI after completing view predictions to allow you to correct any mispredictions. 
# 'adaptive' will only launch a GUI if there are any poor predictions or missing views
# 'automatic' will not launch the GUI and will use the predictions as they are.

slice_info_df, num_phases = select_views(case, case_src, case_dst, MODEL_DIR, states, option, correct_mode, logger)

logger.success(f'View selection complete.')
logger.info(f'Number of phases: {num_phases}')

2026-06-27 10:14:20.342 | INFO     | bivme.preprocessing.dicom.select_views:select_views:142 - Performing metadata-based view prediction...
2026-06-27 10:14:20.534 | INFO     | bivme.preprocessing.dicom.src.viewselection:sort_dicom_per_series:227 - Series 8 contains 3 merged series. Splitting...
2026-06-27 10:14:20.535 | INFO     | bivme.preprocessing.dicom.src.viewselection:sort_dicom_per_series:228 - New 'synthetic' series will range from: 14 to 16
2026-06-27 10:14:20.540 | INFO     | bivme.preprocessing.dicom.src.viewselection:sort_dicom_per_series:227 - Series 13 contains 7 merged series. Splitting...
2026-06-27 10:14:20.540 | INFO     | bivme.preprocessing.dicom.src.viewselection:sort_dicom_per_series:228 - New 'synthetic' series will range from: 17 to 23
2026-06-27 10:14:20.548 | INFO     | bivme.preprocessing.dicom.src.viewselection:sort_dicom_per_series:227 - Series 9 contains 2 merged series. Splitting...
2026-06-27 10:14:20.548 | INFO     | bivme.preprocessing.dicom.src.views

2026-06-27 10:14:33.167 Uvicorn server started on 0.0.0.0:8095



  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8095/biv-me/viewcorrection
  Network URL: http://172.18.0.2:8095/biv-me/viewcorrection
  External URL: http://194.21.48.20:8095/biv-me/viewcorrection

  Stopping...


2026-06-27 10:15:58.209 | INFO     | bivme.preprocessing.dicom.select_views:correct_views_webgui:7 - Web interface closed. Elaborating results...
2026-06-27 10:15:58.212 | INFO     | bivme.preprocessing.dicom.select_views:correct_views_webgui:9 - ----- Saving view predictions...
2026-06-27 10:15:58.213 | INFO     | bivme.preprocessing.dicom.select_views:correct_views_webgui:13 - ----- Series 7 saved to Excluded.
2026-06-27 10:15:58.214 | INFO     | bivme.preprocessing.dicom.select_views:correct_views_webgui:13 - ----- Series 14 saved to Excluded.
2026-06-27 10:15:58.214 | INFO     | bivme.preprocessing.dicom.select_views:correct_views_webgui:13 - ----- Series 15 saved to 4ch.
2026-06-27 10:15:58.215 | INFO     | bivme.preprocessing.dicom.select_views:correct_views_webgui:13 - ----- Series 16 saved to Excluded.
2026-06-27 10:15:58.216 | INFO     | bivme.preprocessing.dicom.select_views:correct_views_webgui:13 - ----- Series 17 saved to Excluded.
2026-06-27 10:15:58.216 | INFO     | bivm

## Step 2: Segmentation
Using the view classifications from step 1, images will be written into nifti form and sent to the segmentation models to be automatically segmented. There is one segmentation model specific to each view (currently SAX, 2ch, 3ch, 4ch, or RVOT), so only images belonging to those classes will be used from now on. 

In [43]:
## Segmentation
seg_start_time = time.time()
segment_views(case_dst, MODEL_DIR, slice_info_df, logger) # Segmentation occurs on 2D+t (3D) images using 3D nnU-Net models
seg_end_time = time.time()

logger.success(f'Segmentation complete. Time taken: {seg_end_time-seg_start_time} seconds.')

2026-06-08 08:12:32.927 | INFO     | bivme.preprocessing.dicom.segment_views:segment_views:95 - Writing SAX images to nifti files...
2026-06-08 08:12:33.080 | INFO     | bivme.preprocessing.dicom.segment_views:segment_views:110 - Segmenting SAX images...


perform_everything_on_device=True is only supported for cuda devices! Setting this to False
There are 7 cases in the source folder
I am processing 0 out of 1 (max process ID is 0, we start counting with 0!)
There are 7 cases that I would like to predict

Predicting SAX_3d_11:
perform_everything_on_device: False


100%|███████████████████████████████████████████████████████| 4/4 [00:52<00:00, 13.07s/it]


sending off prediction to background worker for resampling and export
done with SAX_3d_11

Predicting SAX_3d_12:
perform_everything_on_device: False


100%|███████████████████████████████████████████████████████| 4/4 [00:48<00:00, 12.12s/it]


...

2026-06-08 08:19:18.233 | INFO     | bivme.preprocessing.dicom.segment_views:predict_view:63 - Done with 3ch
2026-06-08 08:19:18.236 | INFO     | bivme.preprocessing.dicom.segment_views:segment_views:95 - Writing 4ch images to nifti files...
2026-06-08 08:19:18.265 | INFO     | bivme.preprocessing.dicom.segment_views:segment_views:110 - Segmenting 4ch images...


perform_everything_on_device=True is only supported for cuda devices! Setting this to False
There are 1 cases in the source folder
I am processing 0 out of 1 (max process ID is 0, we start counting with 0!)
There are 1 cases that I would like to predict

Predicting 4ch_3d_22:
perform_everything_on_device: False


100%|███████████████████████████████████████████████████████| 4/4 [01:01<00:00, 15.26s/it]


sending off prediction to background worker for resampling and export
done with 4ch_3d_22


2026-06-08 08:20:24.209 | INFO     | bivme.preprocessing.dicom.segment_views:predict_view:63 - Done with 4ch
2026-06-08 08:20:24.212 | INFO     | bivme.preprocessing.dicom.segment_views:segment_views:92 - No RVOT images found, skipping...
2026-06-08 08:20:24.215 | SUCCESS  | __main__:<module>:6 - Segmentation complete. Time taken: 471.2886779308319 seconds.


## Step 2.1: Correct for mismatching phases (conditional)
Often, LAX and SAX series do not have matching number of phases. In that case, we need to resample segmentations to have the same number of phases. We use the SAX series as the reference for the 'right' number of phases.

In [44]:
## Resample segmentations if phases are not consistent between SAX and LAX views
correct_phase_mismatch(case_dst, slice_info_df, num_phases, logger)

2026-06-08 08:20:32.395 | SUCCESS  | bivme.preprocessing.dicom.correct_phase_mismatch:correct_phase_mismatch:27 - No phase mismatches found. No resampling required.


## Step 2.2: Review segmentations (optional)
Images and segmentations are stored here in nifti format (.nii.gz). 

    processed_path   
    └───batch_ID
        └───case


If needed, segmentations can be corrected here, and subsequent code executions will incorporate those changes. 3D Slicer or ITKSnap are good tools for correcting segmentations. We are working on a way to automatically load the images and segmentations into Slicer via a Python backend to make this review process easier. 

## Step 3: Generate contours
Contours are generated from the perimeters of the segmentation labels, and other key landmarks (mitral valve, tricuspid valve, aortic valve, pulmonary valve, rv inserts, and LV apex) are estimated from intersections of contours. 2D contours are transformed into 3D cartesian space using the affine constructed from the image position, orientation, and pixel spacing metadata in the DICOMs.

In [45]:
slice_dict = generate_contours(case_dst, slice_info_df, num_phases, logger)
logger.success(f'Guide points generated successfully.')

2026-06-08 08:20:37.795 | INFO     | bivme.preprocessing.dicom.generate_contours:generate_contours:10 - Generating contours for SAX slice 11...
2026-06-08 08:20:37.892 | INFO     | bivme.preprocessing.dicom.generate_contours:generate_contours:10 - Generating contours for SAX slice 12...
2026-06-08 08:20:37.966 | INFO     | bivme.preprocessing.dicom.generate_contours:generate_contours:10 - Generating contours for SAX slice 14...
2026-06-08 08:20:38.075 | INFO     | bivme.preprocessing.dicom.generate_contours:generate_contours:10 - Generating contours for SAX slice 16...
2026-06-08 08:20:38.133 | INFO     | bivme.preprocessing.dicom.generate_contours:generate_contours:10 - Generating contours for SAX slice 17...
2026-06-08 08:20:38.255 | INFO     | bivme.preprocessing.dicom.generate_contours:generate_contours:10 - Generating contours for SAX slice 18...
2026-06-08 08:20:38.311 | INFO     | bivme.preprocessing.dicom.generate_contours:generate_contours:10 - Generating contours for SAX slic

## Step 4: Export guidepoints
Contours are then exported into a standardised form, called a GPFile. There is one of these files per frame. This is what the model fitting part of the code requires. There is also a SliceInfoFile which contains the affine information for each slice, which is necessary to allow for breath hold correction prior to model fitting. 

Temporal smoothing of landmark points is enabled by default, but can be disabled by changing the boolean below. 

In [46]:
smooth_landmarks = False # Set to True to smooth the landmarks, False to keep them as they are
gp_dir = os.path.join(output, case)

export_guidepoints(case_dst, gp_dir, slice_dict, smooth_landmarks)
logger.success(f'Guide points exported successfully.')
logger.success(f'Case {case} complete.')
logger.info(f'Total time taken: {time.time()-start_time} seconds.')

2026-06-08 08:20:42.190 | SUCCESS  | __main__:<module>:5 - Guide points exported successfully.
2026-06-08 08:20:42.192 | SUCCESS  | __main__:<module>:6 - Case sb3101 complete.
2026-06-08 08:20:42.192 | INFO     | __main__:<module>:7 - Total time taken: 681.6121828556061 seconds.


## Step 5 (Optional): Plot guidepoints
The code below will plot the guidepoints at each frame as html files, which can be viewed at:

    plotting_path   
    └───batch_ID
        └───case
            └───html

Here's a good place to find any issues before proceeding to model fitting. 

In [47]:
generate_plots_preprocessing = True # Set to True to generate plots for preprocessing, False to skip plotting
include_images = True # Set to True to include images in the HTML file, False to exclude them. Setting to False will speed up the plotting process, as images take a long time to plot and take up a lot of space in the HTML file.

if generate_plots_preprocessing:
    if include_images:
        image_path = os.path.join(case_dst, 'images')
    else:
        image_path = None

    generate_html(gp_dir, out_dir=plotting, gp_suffix='', si_suffix='', frames_to_fit=[], my_logger=logger, model_path=None, image_path=image_path)

    logger.info(f'Guidepoints plotted at {os.path.join(plotting,case,"html")}.')

2026-06-08 08:20:46.393 | INFO     | bivme.plotting.plot_guidepoints:generate_html:174 - case: sb3101


2026-06-08 08:20:57.112 | INFO     | __main__:<module>:12 - Guidepoints plotted at ../../sunnybrook/processing/testsb9/sb3101/html.


# Start of model fitting

# Step 6: Model fitting
The model fitting takes in the GPFiles and SliceInfoFiles generated during preprocessing to generate biventricular meshes at each frame. An iterative linear least squares fit is applied, as described in Mauger, C., Gilbert, K., Suinesiaputra, A., Pontre, B., Omens, J., McCulloch, A., & Young, A. (2018, July). An iterative diffeomorphic algorithm for registration of subdivision surfaces: application to congenital heart disease. In 2018 40th Annual International Conference of the IEEE Engineering in Medicine and Biology Society (EMBC) (pp. 596-599). IEEE. DOI: 10.1109/EMBC.2018.8512394. 

Breath hold misalignment is corrected by the intersection of short and long axis contours. 

Meshes are generated in the output directory you provided in the config file. Mesh vertices are exported in .txt format by default, but can also be exported with faces in .vtk and .obj formats as well for visualisation and postprocessing purposes.

Unfortunately, there is no opportunity for intervention during this step. 

In [48]:
# Set input for fitting equal to the output of preprocessing
config["input_fitting"]["gp_directory"] = output

# Save a copy of the config file to the output folder
output_folder = Path(config["output_fitting"]["output_directory"])
output_folder.mkdir(parents=True, exist_ok=True)
shutil.copy(config_file, output_folder)

# Force overwrite to true
config["output_fitting"]["overwrite"] = True

# Set whether to generate plots for fitting
config["plotting"]["generate_plots_fitting"] = True
config["plotting"]["include_images"] = True # Set to True to include images in the HTML plots, False to exclude them. Setting to False will speed up the plotting process, as images take a long time to plot and take up a lot of space in the HTML file.

In [49]:
# Where Charlène's magic happens
run_fitting(case, config, logger)

2026-06-08 08:20:57.190 | INFO     | main:run_fitting:26 - Processing sb3101
2026-06-08 08:20:57.195 | INFO     | bivme.fitting.perform_fit:perform_fitting:80 - case: sb3101
2026-06-08 08:20:57.196 | INFO     | bivme.fitting.perform_fit:perform_fitting:88 - ED set to frame #0
2026-06-08 08:20:57.196 | INFO     | bivme.fitting.perform_fit:perform_fitting:105 - Shift measured only at ED frame
2026-06-08 08:20:58.017 | INFO     | bivme.fitting.perform_fit:perform_fitting:204 - Calculating pose and scale sb3101...
2026-06-08 08:20:58.051 | INFO     | bivme.fitting.perform_fit:perform_fitting:209 - Fitting of sb3101
2026-06-08 08:20:58.054 | INFO     | bivme.fitting.perform_fit:perform_fitting:222 - Processing frame #0
2026-06-08 08:20:58.248 | INFO     | bivme.fitting.diffeomorphic_fitting_utils:solve_least_squares_problem:277 -      Iteration 1 Weight 900000000000.0	 ICF error 2.205657065794542
2026-06-08 08:20:58.488 | INFO     | bivme.fitting.diffeomorphic_fitting_utils:solve_least_squa

2026-06-08 08:24:01.840 | INFO     | main:run_fitting:38 - Average residuals: 1.3207822573603052 for case sb3101


In [52]:
from extract_volumetric_mesh import export_volumetric_mesh_meshtool
frame_num = 0
export_volumetric_mesh_meshtool(output_folder, case, frame_num)


Input surface mesh: ../../output-sb/sb3101/vtk/sb3101_EPICARDIAL_000.vtk
Mesh superficie convertita in vtk 42 per meshtool: ../../output-sb/sb3101/vtk/sb3101_EPICARDIAL_000_vtk42.vtk
Input surface mesh: ../../output-sb/sb3101/vtk/sb3101_LV_ENDOCARDIAL_000.vtk
Mesh superficie convertita in vtk 42 per meshtool: ../../output-sb/sb3101/vtk/sb3101_LV_ENDOCARDIAL_000_vtk42.vtk
Input surface mesh: ../../output-sb/sb3101/vtk/sb3101_RV_ENDOCARDIAL_000.vtk
Mesh superficie convertita in vtk 42 per meshtool: ../../output-sb/sb3101/vtk/sb3101_RV_ENDOCARDIAL_000_vtk42.vtk
Generazione mesh volumetrica con meshtool...
Esecuzione comando: meshtool generate mesh -surf=../../output-sb/sb3101/vtk/sb3101_EPICARDIAL_000_vtk42.vtk,../../output-sb/sb3101/vtk/sb3101_LV_ENDOCARDIAL_000_vtk42.vtk,../../output-sb/sb3101/vtk/sb3101_RV_ENDOCARDIAL_000_vtk42.vtk -ins_tag=3,2,1 -outmsh=../../output-sb/sb3101/volumetric/sb3101_volmesh -scale=1.0 -ofmt=vtk_bin


read_mesh_selected warning: No fibers detected. Generating default fibers.
read_mesh_selected warning: No fibers detected. Generating default fibers.
read_mesh_selected warning: No fibers detected. Generating default fibers.


OpenMP thread utilization: 24 / 24 threads.
Reading surface: ../../output-sb/sb3101/vtk/sb3101_EPICARDIAL_000_vtk42
Reading vtk file v.4.2 (3):                            [============] 00h:00m:00s FIN 
Building surface kdtree ..
Computing connectivity graphs:                         [============] 00h:00m:00s FIN 
Reading surface: ../../output-sb/sb3101/vtk/sb3101_LV_ENDOCARDIAL_000_vtk42
Reading vtk file v.4.2 (3):                            [============] 00h:00m:00s FIN 
Computing connectivity graphs:                         [============] 00h:00m:00s FIN 
Unifying surfaces ..
Interface size: 72
Reindexing progress:                                   [============] 00h:00m:00s FIN 
Building surface kdtree ..
Reading surface: ../../output-sb/sb3101/vtk/sb3101_RV_ENDOCARDIAL_000_vtk42
Reading vtk file v.4.2 (3):                            [============] 00h:00m:00s FIN 
Computing connectivity graphs:                         [============] 00h:00m:00s FIN 
Unifying surfaces ..
Interfac

compute_full_mesh_connectivity warning: connectivity read from ../../output-sb/sb3101/volumetric/sb3101_volmesh.fcon is inconsistent! Recomputing it!


Iter 2: splitting 82748 edges, 48.0453 % of total 172229 [============] 00h:00m:01s FIN 
Done in 0.379488 sec
Iter 3: splitting 113265 edges, 27.4562 % of total 412530 [============] 00h:00m:00s FIN 
Done in 0.587034 sec
Iter 4: splitting 79391 edges, 11.0787 % of total 716611 [============] 00h:00m:01s FIN 
Done in 0.546192 sec
Iter 5: splitting 14215 edges, 1.5619 % of total 910096 [============] 00h:00m:00s FIN 
Done in 0.206828 sec
Pass 1: Corrected 372058 inside-out elements.
Pass 2: Corrected 0 inside-out elements.
Computing connectivity graphs:                         [============] 00h:00m:00s FIN 
Computing connectivity graphs:                         [============] 00h:00m:00s FIN 
laplace smoothing progress:                            [============] 00h:00m:00s FIN 
laplace smoothing progress:                            [============] 00h:00m:01s FIN 
Computing surface and surface normals ..
Computing connectivity graphs:                         [============] 00h:00m:00s FI

## (Optional) Analysis and/or visualisation of models
You can visualise the fitted models frame-by-frame in your browser by opening an html in `output_directory\case\html`. Alternatively, if you have a mesh visualisation software, you can load in the .vtk or .obj files you generated to view the meshes beating through time. I recommend Paraview, as it is open source, and has nice camera controls for creating  videos of the meshes beating through time.     

We haven't added any analysis tools to this notebook yet. Refer to the main README on how to generate metrics from the biv-me models. 